# Step 3: Model Design

This notebook walks through the **EncDec-AD** architecture and anomaly scoring methodology
used in this project. We cover the model structure, training loop, reconstruction comparison,
and the full scoring pipeline.

In [ ]:
import sys
from pathlib import Path

# Find project root (works whether jupyter is started from code/, project root, or elsewhere)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists(), (
    f"Could not find project root. CWD={Path.cwd()}\n"
    "Please run: uv run jupyter notebook   (from the project root)"
)
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import logging
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Suppress verbose logging from src modules — we print what we need explicitly
logging.basicConfig(level=logging.WARNING)
for name in ["src.model", "src.scorer", "src.preprocess", "src.synthetic"]:
    logging.getLogger(name).setLevel(logging.WARNING)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Overview

This project implements the **EncDec-AD** model from:

> Malhotra, P., Ramakrishnan, A., Anand, G., Vig, L., Agarwal, P., & Shroff, G. (2016).
> *LSTM-based Encoder-Decoder for Multi-sensor Anomaly Detection.*
> ICML 2016 Anomaly Detection Workshop.

**Core idea:** Train an LSTM encoder-decoder to reconstruct *normal* time series data.
At inference time, anomalous patterns produce higher reconstruction error because the
model has only learned the structure of normal behaviour. We fit a distribution on the
reconstruction errors from normal data and use it to derive per-point anomaly scores.

## 2. Architecture

The model has three components:

1. **Encoder LSTM** -- reads the input sequence and compresses it into a fixed-size hidden state.
2. **Decoder LSTM** -- receives the encoder's hidden state and reconstructs the sequence in *reverse* order.
3. **Linear output layer** -- maps decoder hidden states back to the input feature space.

In [ ]:
from src.model import EncDecAD, ModelConfig, create_model

model = create_model(
    input_dim=1,
    hidden_dim=64,
    num_layers=1,
    dropout=0.2,
    sequence_length=336,
)

print("=== Model Architecture ===")
print(model)
print()
print("=== Config ===")
for k, v in model.get_config().items():
    print(f"  {k}: {v}")
print(f"\nTotal trainable parameters: {model.count_parameters():,}")

In [ ]:
# Trace dimensions through a forward pass with dummy data
batch, seq_len, features = 1, 336, 1
x = torch.randn(batch, seq_len, features)

print(f"Input shape:           {tuple(x.shape)}")

# Encoder
h_n, c_n = model.encode(x)
print(f"Encoder h_n shape:     {tuple(h_n.shape)}   (num_layers, batch, hidden_dim)")
print(f"Encoder c_n shape:     {tuple(c_n.shape)}")

# Full forward pass (includes reverse reconstruction)
x_hat = model(x)
print(f"Output shape:          {tuple(x_hat.shape)}")
print(f"\nInput == Output shape:  {x.shape == x_hat.shape}")

## 3. Reverse Reconstruction

A key design choice from the paper: the decoder reconstructs the sequence in **reverse order**
(`x'(L), x'(L-1), ..., x'(1)`).

**Why reverse?** If reconstruction were in the forward direction, the decoder could rely on
short-term memory of recent values to produce good reconstructions without truly compressing
the entire sequence. By reversing the target, the first value the decoder must produce
(`x(L)`, the last input timestep) is furthest from the encoder's most recent memory. This
forces the encoder to build a holistic representation of the *full* sequence in its hidden
state, rather than just memorising the tail end.

In `EncDecAD.forward()`:
1. The input is flipped along the time axis before being fed to the decoder.
2. The decoder output is flipped back so the loss is computed in the original order.

## 4. Teacher Forcing

During both training and inference the decoder receives the **actual (reversed) input** at each
timestep, not its own previous predictions. This is called *teacher forcing*.

In a generative model we would typically anneal away teacher forcing so the model learns to
be self-consistent. But for anomaly detection the goal is different: we want a **consistent
measure of reconstruction error**. If the decoder consumed its own (possibly drifting)
predictions, error at timestep `t` would depend on errors at earlier timesteps, making the
per-point error signal noisy and hard to calibrate.

By always feeding ground truth, each point's reconstruction error reflects the model's
difficulty with *that specific local context*, independent of errors elsewhere in the window.

## 5. Training Loop

Below we load the NYC taxi data, build the model, and train for a small number of epochs
(10) to demonstrate the loop. A full run uses ~100 epochs with early stopping.

In [ ]:
from src.preprocess import preprocess_pipeline

DATA_PATH = str(PROJECT_ROOT / "data" / "nyc_taxi.csv")

dataloaders, normalized_splits, scaler, week_info, split_indices = preprocess_pipeline(
    DATA_PATH, batch_size=4
)

print(f"Train: {len(dataloaders['train'].dataset)} weeks ({len(dataloaders['train'])} batches)")
print(f"Val:   {len(dataloaders['val'].dataset)} weeks ({len(dataloaders['val'])} batches)")
print(f"Test:  {len(dataloaders['test'].dataset)} weeks")
print(f"Scaler mean: {scaler.mean_[0]:.0f}, std: {scaler.scale_[0]:.0f}")

In [ ]:
# Build a fresh model and train for 10 demo epochs
model = create_model(hidden_dim=64, num_layers=1, dropout=0.2)
model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

EPOCHS = 10
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for batch in dataloaders["train"]:
        x = batch.to(device)
        optimizer.zero_grad()
        x_hat = model(x)
        loss = criterion(x_hat, x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)

    # --- Validate ---
    model.eval()
    v_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for batch in dataloaders["val"]:
            x = batch.to(device)
            x_hat = model(x)
            v_loss += criterion(x_hat, x).item()
            n_val += 1
    val_losses.append(v_loss / n_val)

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train: {train_losses[-1]:.6f} | Val: {val_losses[-1]:.6f}")

print("\nTraining complete.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, marker="o", label="Train loss")
ax.plot(range(1, EPOCHS + 1), val_losses, marker="s", label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Reconstruction Comparison

After training, we compare how well the model reconstructs a **normal** week versus an
**anomaly** week. The model should reconstruct normal patterns accurately and struggle with
anomalous ones, producing higher error.

In [ ]:
# Find one normal and one anomaly week from the test set
test_indices = split_indices["test"]
normal_test_idx = None
anomaly_test_idx = None

for i, idx in enumerate(test_indices):
    if week_info[idx]["is_anomaly"] and anomaly_test_idx is None:
        anomaly_test_idx = i
    elif not week_info[idx]["is_anomaly"] and normal_test_idx is None:
        normal_test_idx = i
    if normal_test_idx is not None and anomaly_test_idx is not None:
        break

test_data = normalized_splits["test"]


def reconstruct(model, data_1d, device):
    """Reconstruct a single 1-D sequence and return original, reconstruction, error."""
    x = torch.FloatTensor(data_1d).unsqueeze(0).unsqueeze(-1).to(device)
    model.eval()
    with torch.no_grad():
        x_hat = model(x)
    orig = x.squeeze().cpu().numpy()
    recon = x_hat.squeeze().cpu().numpy()
    error = np.abs(orig - recon)
    return orig, recon, error


fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
time = np.arange(336) / 48  # Convert to days

for row, (idx, label) in enumerate(
    [(normal_test_idx, "Normal Week"), (anomaly_test_idx, "Anomaly Week")]
):
    if idx is None:
        for col in range(3):
            axes[row, col].text(0.5, 0.5, f"No {label.lower()} found",
                                ha="center", va="center", transform=axes[row, col].transAxes)
        continue

    week_label = week_info[test_indices[idx]]["year_week"]
    orig, recon, error = reconstruct(model, test_data[idx], device)

    axes[row, 0].plot(time, orig, label="Original")
    axes[row, 0].set_title(f"{label} ({week_label}) -- Original")
    axes[row, 0].set_ylabel("Normalized value")

    axes[row, 1].plot(time, orig, alpha=0.5, label="Original")
    axes[row, 1].plot(time, recon, label="Reconstructed")
    axes[row, 1].set_title(f"{label} -- Reconstructed")
    axes[row, 1].legend()

    axes[row, 2].fill_between(time, error, alpha=0.6, color="red")
    axes[row, 2].set_title(f"{label} -- |Error|  (mean={error.mean():.4f})")
    axes[row, 2].set_ylabel("|error|")

for ax in axes[1]:
    ax.set_xlabel("Day within week")

plt.tight_layout()
plt.show()

## 7. Anomaly Scoring

The scoring methodology from Malhotra et al. (2016) works as follows:

1. **Fit a normal error distribution.** Compute reconstruction errors on a *normal* validation
   set and fit a Gaussian `N(mu, sigma)` to the per-point absolute errors.

2. **Per-point anomaly score.** For each point `i` in a window:
   ```
   a_i = (e_i - mu)^2 / sigma^2
   ```
   This is the squared z-score -- how many standard deviations the error deviates from normal.

3. **Point threshold (tau).** Set `tau` to the 99.99th percentile of per-point scores computed
   on normal validation data. Points above `tau` are considered anomalous.

4. **HardCriterion (window decision).** A window is flagged as anomalous if at least
   `k = 5` individual points exceed `tau`. This avoids false alarms from isolated spiky points.

In [ ]:
from src.scorer import AnomalyScorer, ScorerConfig

scorer_config = ScorerConfig(
    scoring_mode="point",
    threshold_percentile=99.99,
    hard_criterion_k=5,
)
scorer = AnomalyScorer(config=scorer_config)

# Fit on the validation (normal) data
scorer.fit(model, dataloaders["val"], device)

print(f"mu (mean abs error on normal data): {scorer.mu_point[0]:.6f}")
print(f"sigma^2 (variance):                 {scorer.sigma_point[0]:.6f}")

In [ ]:
# Compute point scores on validation data and set the threshold
val_point_scores, val_window_scores, _ = scorer.compute_point_scores(
    model, dataloaders["val"], device
)
tau = scorer.set_point_threshold(val_point_scores)
print(f"Point threshold tau (99.99th percentile): {tau:.4f}")

# Score the test set
test_point_scores, test_window_scores, _ = scorer.compute_point_scores(
    model, dataloaders["test"], device
)

# Plot score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(val_point_scores.flatten(), bins=80, alpha=0.7, label="Normal (val)", density=True)
axes[0].axvline(tau, color="red", ls="--", label=f"tau = {tau:.2f}")
axes[0].set_title("Per-Point Score Distribution (Normal Validation)")
axes[0].set_xlabel("Anomaly score")
axes[0].set_ylabel("Density")
axes[0].legend()

# Separate test scores by normal vs anomaly weeks
test_info_list = [week_info[i] for i in split_indices["test"]]
normal_mask = np.array([not w["is_anomaly"] for w in test_info_list])
anomaly_mask = ~normal_mask

if normal_mask.any():
    axes[1].hist(test_point_scores[normal_mask].flatten(), bins=80, alpha=0.6,
                 label="Normal test", density=True)
if anomaly_mask.any():
    axes[1].hist(test_point_scores[anomaly_mask].flatten(), bins=80, alpha=0.6,
                 label="Anomaly test", density=True)
axes[1].axvline(tau, color="red", ls="--", label=f"tau = {tau:.2f}")
axes[1].set_title("Per-Point Score Distribution (Test Set)")
axes[1].set_xlabel("Anomaly score")
axes[1].legend()

plt.tight_layout()
plt.show()

# Apply HardCriterion
point_preds = scorer.predict_points(test_point_scores)
window_preds = scorer.predict_windows_from_points(point_preds)

print(f"\nHardCriterion (k={scorer.config.hard_criterion_k}):")
for i, (w, pred) in enumerate(zip(test_info_list, window_preds)):
    anomalous_pts = int(point_preds[i].sum())
    status = "ANOMALY" if pred else "normal"
    actual = "ANOMALY" if w["is_anomaly"] else "normal"
    match = "Y" if (pred == w["is_anomaly"]) else "N"
    print(f"  {w['year_week']}  anomalous_pts={anomalous_pts:3d}  -> {status:<8s}  (actual: {actual})  {match}")

## 8. Anomaly Localization

Once a week-long window is flagged as anomalous, we want to narrow down *where* within the
week the anomaly occurs. The localization method uses a **6-hour sliding window**:

1. Slide a 6-hour window (12 samples at 30-min resolution) across the 336-point week.
2. At each position, count the number of points whose anomaly score exceeds the threshold,
   and check for score spikes.
3. The sub-window with the highest concentration of anomalous points is reported as the
   anomaly location.

This gives operators a focused time range to investigate rather than an entire week.

In [ ]:
from src.preprocess import get_test_timestamps

test_timestamps = get_test_timestamps(week_info, split_indices)

# Localize anomalies in flagged windows
for i, (pred, info) in enumerate(zip(window_preds, test_info_list)):
    if not pred:
        continue

    result = scorer.localize_anomaly(test_point_scores[i], test_timestamps[i])
    print(f"Week {info['year_week']} (actual: {'ANOMALY' if info['is_anomaly'] else 'normal'}):")
    print(f"  Localized to: {result['anomaly_start']} -- {result['anomaly_end']}")
    print(f"  Scale: {result['scale_hours']}h ({result['scale_samples']} samples)")
    print(f"  Contrast ratio: {result['contrast_ratio']:.2f}")
    print()

## 9. Summary

The full anomaly detection pipeline:

| Step | Action | Key detail |
|------|--------|------------|
| **Preprocess** | Segment into weekly windows, normalize | Fit scaler on training data only |
| **Train** | Encoder-decoder on normal data | MSE loss, reverse reconstruction, teacher forcing |
| **Fit scorer** | Learn `N(mu, sigma)` from normal reconstruction errors | Per-point scoring |
| **Detect** | Score test windows, apply threshold + HardCriterion | `tau` = 99.99th pctl, `k >= 5` |
| **Localize** | 6-hour sliding window within flagged weeks | Find highest anomaly concentration |

The next steps (`4_train_model.py`, `5_evaluate_model.py`) run this pipeline end-to-end
with full hyperparameters and evaluation metrics.